# Módulo 4: Aprendizaje Automático (Machine Learning)
## Semana 3 - Clase 07: Modelos de Ensamble (La Sabiduría Colectiva)

Hasta ahora hemos confiado el destino de nuestro negocio a un solo modelo matemático (un Árbol de Decisión o una Regresión). El problema radica en que los modelos individuales pueden sufrir de **Overfitting** (memorizan el ruido) o **Underfitting** (son demasiado simples).

Hoy introduciremos la estrategia más poderosa del Machine Learning moderno: **Los Ensambles**. Combinaremos cientos de modelos débiles para crear predictores corporativos infalibles.

---
## **Sección 1: El Caso de Negocio - Fuga de Clientes (Churn)**

Supongamos que somos la división de Analítica de una empresa de Telecomunicaciones. Nuestro problema: Los clientes están cancelando sus suscripciones (Churn) y perdiéndose hacia la competencia. 

**Misión:** Construir un modelo predictivo que analice el comportamiento actual del cliente e infiera (prediga) si está en riesgo inminente de cancelación. Con esta alerta, Marketing podrá ofrecerles promociones preventivas.

Generaremos un dataset sintético que simula el comportamiento de 5,000 clientes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from IPython.display import display

# 1. Generación de datos corporativos simulados
X_raw, y_raw = make_classification(
    n_samples=5000, 
    n_features=5, 
    n_informative=4, 
    n_redundant=1, 
    random_state=42,
    weights=[0.7, 0.3] # 70% clientes leales, 30% abandonos (churn)
)

columnas = ['facturacion_mensual', 'antiguedad_meses', 'llamadas_soporte', 'retrasos_pago', 'consumo_datos_gb']
df_clientes = pd.DataFrame(X_raw, columns=columnas)

# Escalar los datos a rangos más realistas para efectos visuales
df_clientes['facturacion_mensual'] = np.abs(df_clientes['facturacion_mensual'] * 20 + 60)
df_clientes['antiguedad_meses'] = np.abs(df_clientes['antiguedad_meses'] * 12 + 24).astype(int)
df_clientes['llamadas_soporte'] = np.abs(df_clientes['llamadas_soporte'] * 2 + 1).astype(int)
df_clientes['consumo_datos_gb'] = np.abs(df_clientes['consumo_datos_gb'] * 50 + 100)

df_clientes['abandono'] = y_raw

display(df_clientes.head())

### **Pausa Analítica: ¿Cómo están correlacionados estos datos?**
Aunque los datos son sintéticos, el algoritmo inyectó reglas lógicas (correlaciones) matemáticas ocultas.

Vamos a visualizar la matriz de correlación y un gráfico de dispersión para entender visualmente cómo ciertas variables empujan al cliente hacia la zona de peligro de Abandono (Puntos rojos).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Mapa de Calor de Correlaciones
sns.heatmap(df_clientes.corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=axes[0])
axes[0].set_title("Matriz de Correlación")

# 2. Gráfico de Dispersión
sns.scatterplot(
    data=df_clientes, 
    x='facturacion_mensual', 
    y='retrasos_pago', 
    hue='abandono', 
    palette=['green', 'red'], 
    alpha=0.6, 
    ax=axes[1]
)
axes[1].set_title("Dispersión: Facturación vs Retrasos de Pago")

plt.tight_layout()
plt.show()

# Nota: Observa cómo los puntos rojos (Abandono=1) tienden a agruparse en ciertas zonas del gráfico.

### **Extracción de Tuplas Reales (Filtros de Pandas)**
Para comprobar que el gráfico no nos engaña, vamos a extraer directamente 10 clientes (tuplas) usando filtros de Pandas que representan los perfiles extremos discutidos en la teoría.

In [ ]:
from IPython.display import display

print("--- CASO 1: PERFIL CRÍTICO DE ABANDONO (CHURN = 1) ---")
print("Buscamos a los clientes que ya abandonaron y los ordenamos por mayores retrasos de pago y llamadas de soporte.")
abandono_real = df_clientes[df_clientes['abandono'] == 1].sort_values(by=['retrasos_pago', 'llamadas_soporte'], ascending=[False, False])
display(abandono_real.head(5))

print("\n--- CASO 2: PERFIL DE LEALTAD ABSOLUTA (CHURN = 0) ---")
print("Filtramos a clientes fieles y los ordenamos por su antigüedad en la empresa.")
lealtad_real = df_clientes[df_clientes['abandono'] == 0].sort_values(by=['antiguedad_meses'], ascending=[False])
display(lealtad_real.head(5))

---
## **Sección 2: El Modelo Solitario (El "Lobo Solitario")**

Primero, intentaremos predecir el abandono utilizando un simple Árbol de Decisión. Este será nuestra línea base (Baseline) contra la cual mediremos el poder de los ensambles.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Separación de variables predictoras (X) y vector objetivo (y)
X = df_clientes.drop('abandono', axis=1)
y = df_clientes['abandono']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    
# Entrenamos el "Experto Solitario"
arbol_individual = DecisionTreeClassifier(random_state=42, max_depth=5)
arbol_individual.fit(X_train, y_train)

pred_arbol = arbol_individual.predict(X_test)
acc_arbol = accuracy_score(y_test, pred_arbol)

print(f"Exactitud del Árbol Individual: {acc_arbol * 100:.2f}%")
print("\nReporte Médico-Financiero:\n", classification_report(y_test, pred_arbol))

---
## **Sección 3: Bagging (Random Forest) - El Consejo de Sabios**

El algoritmo **Random Forest** (Bosque Aleatorio) no confía en un solo árbol. Su estrategia es crear, por ejemplo, 100 árboles de decisión en paralelo. A cada árbol le entrega un subconjunto de datos ligeramente diferente (Bootstrapping). Al final, los 100 árboles votan de forma democrática. Esto estabiliza brutalmente la varianza.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Instanciamos el Bosque (100 árboles de decisión trabajando juntos)
bosque = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
bosque.fit(X_train, y_train)

pred_bosque = bosque.predict(X_test)
acc_bosque = accuracy_score(y_test, pred_bosque)

print(f"Exactitud del Random Forest: {acc_bosque * 100:.2f}%")
print("\nReporte Médico-Financiero:\n", classification_report(y_test, pred_bosque))

### **Pausa Analítica de Negocio: Importancia de Variables (Feature Importance)**
Un directivo no solo quiere saber si el modelo funciona, quiere saber **POR QUÉ** los clientes se van. Random Forest nos permite auditar matemáticamente qué variables causaron el impacto mayoritario.

In [ ]:
# Extracción de la importancia matemática de cada variable
importancias = bosque.feature_importances_

# Visualización corporativa
plt.figure(figsize=(8, 4))
sns.barplot(x=importancias, y=X.columns, palette='viridis')
plt.title('Importancia de Variables - Auditoría Random Forest', fontsize=14)
plt.xlabel('Peso en la Decisión')
plt.ylabel('Atributo del Cliente')
plt.show()

# CONCLUSIÓN: El departamento de Marketing ahora sabe exactamente qué medir para lanzar promociones preventivas.

---
## **Sección 4: Boosting (XGBoost) - Aprendizaje Evolutivo**

A diferencia del Bosque que entrena 100 árboles en paralelo, el **Boosting** entrena los árboles de forma estrictamente secuencial. 

1. El **Árbol 1** hace sus predicciones. Comete errores.
2. El **Árbol 2** analiza EXACTAMENTE a qué clientes el Árbol 1 falló, y concentra toda su energía matemática en corregir esos fallos.
3. El **Árbol 3** corrige los errores del Árbol 2, y así sucesivamente.

Para esto utilizaremos `XGBoost`, el algoritmo más galardonado en competencias de Inteligencia Artificial.

In [ ]:
# pip install xgboost (Si no está instalado)
from xgboost import XGBClassifier

# Instanciamos la máquina evolutiva
modelo_xgb = XGBClassifier(
    n_estimators=100, 
    max_depth=4, 
    learning_rate=0.1,  # Tamaño de los pasos de corrección entre árbol y árbol
    random_state=42
)

modelo_xgb.fit(X_train, y_train)

pred_xgb = modelo_xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, pred_xgb)

print(f"Exactitud del XGBoost Evolutivo: {acc_xgb * 100:.2f}%")
print("\nReporte Médico-Financiero:\n", classification_report(y_test, pred_xgb))